# Electron-Aware ANI Energy Workflow

This notebook is the runnable front door for the proposal work. It downloads a TorchANI dataset, prepares train/validation batches, trains a standard ANI baseline and the hydrogen-like radial descriptor model, evaluates energy RMSE/MAE, and plots the comparison.

By default this uses TorchANI `TestData` so **Run All** finishes quickly. The optional ANI1x cell near the end can be enabled for a larger proposal-scale benchmark.

## 1. Confirm Local TorchANI Source

The project is configured to use the inspectable TorchANI checkout under `vendor/torchani`, not a hidden site-packages copy.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "mod_ani").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from mod_ani.local_torchani import use_local_torchani, torchani_source_path

local_torchani = use_local_torchani()
import torchani

print("Project root:", PROJECT_ROOT)
print("Expected TorchANI source:", torchani_source_path())
print("Imported torchani from:", torchani.__file__)

## 2. Configure A Quick Energy Benchmark

This configuration is intentionally small. Increase `max_epochs`, use a larger `batch_size`, or switch to `ani1x_config()` for a larger run.

In [ ]:
from mod_ani.config import quick_test_config

config = quick_test_config(
    max_epochs=2,
    batch_size=128,
    cache_batches=True,
    refresh_batches=False,
)
config

## 3. Download/Open Dataset

TorchANI handles checksums and reuse. Re-running this cell will reuse the existing dataset once downloaded.

In [ ]:
from mod_ani.data import describe_dataset, download_dataset

dataset = download_dataset(config)
describe_dataset(dataset)

## 4. Prepare Train/Validation Batches

Batches are stored under `data/batched/...` and reused unless `config.refresh_batches = True`.

In [ ]:
from mod_ani.data import prepare_batched_dataset

batched = prepare_batched_dataset(dataset, config)
{name: len(split) for name, split in batched.items()}

## 5. Benchmark Available Devices

This cell times a few train-like TorchANI steps on each available backend, then sets `config.device` to the fastest measured option. Small batches can be faster on CPU than MPS/GPU.

In [ ]:
from dataclasses import replace
import time
import torch

from mod_ani.models import build_model
from mod_ani.training import filter_energy_outliers, move_batch


def _sync_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()


def benchmark_device(device_name, steps=3):
    bench_config = replace(config, device=device_name, model_kind="electron_radial")
    device = torch.device(device_name)
    model = build_model(bench_config).to(device=device, dtype=torch.float32)
    optimizer = torch.optim.AdamW(
        model.neural_networks.parameters(),
        lr=bench_config.effective_learning_rate(),
        weight_decay=bench_config.weight_decay,
    )
    source_batch = next(iter(batched["training"].as_dataloader(num_workers=0, pin_memory=False)))
    batch = move_batch(source_batch, device, torch.float32)
    batch = filter_energy_outliers(batch, bench_config.max_abs_energy_hartree)
    if batch is None:
        raise RuntimeError("Benchmark batch only contained filtered energy outliers")

    species = batch["species"]
    coordinates = batch["coordinates"]
    target_energies = batch["energies"]
    num_atoms = (species >= 0).sum(dim=1, dtype=target_energies.dtype)

    # Warm-up step, especially useful for MPS/CUDA lazy setup.
    predicted = model((species, coordinates)).energies
    loss = ((predicted - target_energies).pow(2) / num_atoms.sqrt()).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    _sync_device(device)

    start = time.perf_counter()
    for _ in range(steps):
        predicted = model((species, coordinates)).energies
        loss = ((predicted - target_energies).pow(2) / num_atoms.sqrt()).mean()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.neural_networks.parameters(), config.max_grad_norm)
        optimizer.step()
    _sync_device(device)
    elapsed = time.perf_counter() - start
    return elapsed / steps


candidate_devices = ["cpu"]
if torch.cuda.is_available():
    candidate_devices.append("cuda")
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    candidate_devices.append("mps")

timings = {}
for device_name in candidate_devices:
    try:
        timings[device_name] = benchmark_device(device_name)
    except Exception as exc:
        timings[device_name] = float("inf")
        print(f"{device_name}: unavailable for this benchmark ({exc})")

best_device = min(timings, key=timings.get)
config.device = best_device
print("Seconds per train step:", timings)
print("Selected device:", config.device)

## 6. Train Baseline ANI and Electron-Radial ANI

The baseline uses standard TorchANI radial/angular AEVs. The electron-aware version replaces the radial term with hydrogen-like radial density features and keeps the standard ANI angular term for this Aim 1 benchmark.

In [ ]:
from mod_ani.training import train_pair

histories = train_pair(config, batched)
{kind: history[-1] for kind, history in histories.items()}

## 7. Plot Validation Energy Error

In [ ]:
from pathlib import Path
from mod_ani.plotting import plot_history

fig, ax = plot_history(histories, output=Path("reports/figures/quick_test_rmse.png"))
fig

## 8. Inspect The Descriptor Modification

This cell prints the custom radial descriptor and the local TorchANI AEVComputer that consumes it.

In [ ]:
from mod_ani.descriptors import make_hydrogen_like_aev

aev = make_hydrogen_like_aev(num_species=len(config.symbols))
print(aev)
print("Radial term:", aev.radial)
print("TorchANI AEVComputer source:", local_torchani / "torchani" / "aev" / "_computer.py")
print("TorchANI term source:", local_torchani / "torchani" / "aev" / "_terms.py")

## 9. Optional Larger ANI1x Run

Set `RUN_FULL_ANI1X = True` when you are ready for a larger DFT energy benchmark. This can take substantially longer and use much more disk/RAM than the default test run.

In [ ]:
RUN_FULL_ANI1X = False

if RUN_FULL_ANI1X:
    from mod_ani.config import ani1x_config
    from mod_ani.data import download_dataset, prepare_batched_dataset
    from mod_ani.training import train_pair
    from mod_ani.plotting import plot_history

    full_config = ani1x_config(
        max_epochs=20,
        batch_size=2560,
        cache_batches=False,
        device=config.device,
        verbose=True,
    )
    print("Using device for ANI1x run:", full_config.device)
    print("ANI1x epochs:", full_config.max_epochs)
    print("ANI1x batch size:", full_config.batch_size)
    full_dataset = download_dataset(full_config)
    full_batched = prepare_batched_dataset(full_dataset, full_config)
    full_histories = train_pair(full_config, full_batched)
    plot_history(full_histories, output=Path("reports/figures/ani1x_rmse.png"))
else:
    print("Skipping full ANI1x run. Set RUN_FULL_ANI1X = True to enable it.")